# 01 Baseline and Error Analysis

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP_001_SVC_Normal_Baseline.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)
- [`EXP_002_SVC_Normal_Error_Analysis.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_002_SVC_Normal_Error_Analysis.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '1d47c9dc18348c70f40bdeb82f807a177dd59737'}, '05_model_optimize/SVC/Normal/scripts/EXP_002_SVC_Normal_Error_Analysis.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '86895e5ca70b5ee7dd4cc9dd474a004c427e750d'}}

## EXP_001_SVC_Normal_Baseline

원본: [05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py)  
Git blob: `1d47c9dc18348c70f40bdeb82f807a177dd59737`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

SVC Normal EXP-001: official validation + four-writer Hand-made development evaluation.

From the repository root:
 python 05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py
 python 05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py --pilot

Only the official Training/Validation split is read; official Test is never accessed.
Uses the repository's own SVC build_baseline() when available; fails rather than
silently substituting a different baseline if no SVC base

CLI 인자 (정적 추출): `--pilot`

In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC Normal EXP-001: official validation + four-writer Hand-made development evaluation.

From the repository root:
 python 05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py
 python 05_model_optimize/SVC/Normal/scripts/EXP_001_SVC_Normal_Baseline.py --pilot

Only the official Training/Validation split is read; official Test is never accessed.
Uses the repository's own SVC build_baseline() when available; fails rather than
silently substituting a different baseline if no SVC baseline is found.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import importlib.util
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

ROOT = Path(__file__).resolve().parents[4]
COMMON_DIR = ROOT / '03_experiments' / 'model_optimization'
sys.path.insert(0, str(COMMON_DIR))
from common import fixed_split, pipeline, LABELS

WRITERS = (41, 42, 43, 44)
EXPECTED_SHA = {
    41: '0a829172a72787c770139c73de34cc8b751181829ff6fe08ec260247b9be5a2f',
    42: 'f364fb453d05b4fe2a705c2b8a80683b404608c7c257d664e3ec7ac86932d4c5',
    43: '10db20997c727c1bc1f1299f8e89626fadd3ec9618468d6759c86b0d2ddc4a66',
    44: '0e266a36e7b29edfcca0514836b5c383d13939948b4e335d70f61bb40bdc084d',
}
OUT_BASE = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'results'


def csv_save(path, rows, fields=None):
    if fields is None:
        fields = list(rows[0]) if rows else []
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def load_writer(writer):
    p = ROOT / '02_data' / 'Hand_made' / f'ml26_{writer}.npz'
    digest = hashlib.sha256(p.read_bytes()).hexdigest()
    if digest != EXPECTED_SHA[writer]:
        raise RuntimeError(f'Hand-made SHA256 mismatch for writer {writer}: {digest}')
    with np.load(p, allow_pickle=False) as d:
        if set(d.files) != {'img', 'label', 'label_id'}:
            raise ValueError(f'Unexpected keys in {p}: {d.files}')
        x = d['img']
        y = d['label_id'].astype(np.int64)
        labels = d['label']
    if x.shape != (640, 28, 28) or x.dtype != np.uint8 or y.shape != (640,):
        raise ValueError(f'Unexpected shape/dtype: {p}')
    if not np.array_equal(y, np.array([LABELS.index(str(t)) for t in labels])):
        raise ValueError(f'Label mapping mismatch: {p}')
    if not np.all(np.bincount(y, minlength=16) == 40):
        raise ValueError(f'Writer {writer} not balanced by class')
    return x.reshape(640, 784).astype(np.float32), y, digest


def baseline():
    # The team's original 03_experiments baseline is authoritative.
    # Do not silently replace it with sklearn defaults.
    candidates = [COMMON_DIR / 'svc.py', COMMON_DIR / 'svm.py']
    for p in candidates:
        if p.is_file():
            spec = importlib.util.spec_from_file_location('_project_svc_baseline', p)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            if not hasattr(module, 'build_baseline'):
                continue
            result = module.build_baseline()
            if not (isinstance(result, tuple) and len(result) == 2):
                raise TypeError(f'{p}: build_baseline() should return (estimator, scaled)')
            estimator, scaled = result
            if not isinstance(estimator, SVC):
                raise TypeError(f'{p}: expected sklearn.svm.SVC, got {type(estimator)}')
            if not isinstance(scaled, (bool, np.bool_)):
                raise TypeError(f'{p}: expected boolean scaling flag')
            return estimator, bool(scaled), str(p.relative_to(ROOT))
    raise FileNotFoundError(
        'SVC baseline not found. Expected 03_experiments/model_optimization/svc.py '
        'or svm.py defining build_baseline() -> (SVC, scaled). '
        'Please check your repository for the original SVC baseline file.'
    )


def report(y_true, y_pred):
    return {'n': int(len(y_true)), 'correct': int((y_true == y_pred).sum()),
            'accuracy': float(accuracy_score(y_true, y_pred)),
            'macro_f1': float(f1_score(y_true, y_pred, labels=np.arange(16), average='macro', zero_division=0))}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--pilot', action='store_true', help='Stratified 8000 training / 3200 validation pilot; not the full baseline')
    args = ap.parse_args()
    suffix = 'EXP_001_SVC_Normal_PILOT' if args.pilot else 'EXP_001_SVC_Normal_Baseline'
    out = OUT_BASE / suffix

    base, scaled, source = baseline()
    print('SVC original baseline:', source, flush=True)
    print('SVC parameters:', base.get_params(), flush=True)
    print('Scaling:', scaled, flush=True)

    xtr, ytr, xval, yval, data_sha = fixed_split()
    if len(ytr) != 76800 or len(yval) != 19200:
        raise RuntimeError('Official split mismatch; expected 76800/19200')
    if args.pilot:
        ids, _ = train_test_split(np.arange(len(ytr)), train_size=8000, stratify=ytr, random_state=42)
        val_ids, _ = train_test_split(np.arange(len(yval)), train_size=3200, stratify=yval, random_state=42)
        xtr, ytr, xval, yval = xtr[ids], ytr[ids], xval[val_ids], yval[val_ids]
    print(f'Official Training={len(ytr)}, Validation={len(yval)}. Official Test NOT loaded.', flush=True)

    handmade = {w: load_writer(w) for w in WRITERS}
    clf = pipeline(base, scaled)
    t0 = time.perf_counter()
    print('[SVC Normal] fitting...', flush=True)
    clf.fit(xtr, ytr)
    fit_seconds = time.perf_counter() - t0
    print(f'fit_seconds={fit_seconds:.2f}', flush=True)

    pv = clf.predict(xval).astype(np.int64)
    official = report(yval, pv)
    print(f"Official Validation Accuracy={official['accuracy']:.6f}, F1={official['macro_f1']:.6f}", flush=True)
    truth, predictions, writer_rows, pred_rows = [], [], [], []
    for w in WRITERS:
        x, y, _ = handmade[w]
        p = clf.predict(x).astype(np.int64)
        m = report(y, p)
        writer_rows.append({'writer': w, **m})
        print(f"Writer {w}: {m['correct']}/640, Accuracy={m['accuracy']:.6f}, F1={m['macro_f1']:.6f}", flush=True)
        truth.append(y)
        predictions.append(p)
        pred_rows.extend({'writer': w, 'index': i, 'true': LABELS[int(a)],
                          'prediction': LABELS[int(b)], 'correct': int(a == b)}
                         for i, (a, b) in enumerate(zip(y, p)))
    yt, yp = np.concatenate(truth), np.concatenate(predictions)
    overall = report(yt, yp)
    print(f"Hand-made DEVELOPMENT: Accuracy={overall['accuracy']:.6f}, F1={overall['macro_f1']:.6f} ({overall['correct']}/2560)", flush=True)
    cm = confusion_matrix(yt, yp, labels=np.arange(16))
    pr, re, f1, support = precision_recall_fscore_support(yt, yp, labels=np.arange(16), zero_division=0)
    out.mkdir(parents=True, exist_ok=True)
    csv_save(out / 'writer_results.csv', writer_rows)
    csv_save(out / 'handmade_predictions.csv', pred_rows)
    csv_save(out / 'handmade_confusion_matrix.csv',
             [{'true\\pred': LABELS[i], **{LABELS[j]: int(cm[i, j]) for j in range(16)}} for i in range(16)])
    csv_save(out / 'per_class.csv',
             [{'class': LABELS[i], 'precision': float(pr[i]), 'recall': float(re[i]),
               'f1': float(f1[i]), 'support': int(support[i])} for i in range(16)])
    metrics = {'experiment': suffix, 'model_source': source, 'hyperparameters': base.get_params(),
               'scaled': scaled, 'official_train_n': len(ytr), 'official_val_n': len(yval),
               'official_validation': official, 'handmade_development': overall,
               'writers': writer_rows, 'fit_seconds': fit_seconds,
               'dataset_sha256': data_sha,
               'handmade_sha256': {str(w): handmade[w][2] for w in WRITERS},
               'official_test_used': False,
               'evaluation_role': 'repeated-use Hand-made development; NOT independent final test',
               'environment': {'python': platform.python_version(), 'numpy': np.__version__,
                               'sklearn': sklearn.__version__}}
    (out / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    print('Saved:', out, flush=True)


if __name__ == '__main__':
    main()


## EXP_002_SVC_Normal_Error_Analysis

원본: [05_model_optimize/SVC/Normal/scripts/EXP_002_SVC_Normal_Error_Analysis.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/scripts/EXP_002_SVC_Normal_Error_Analysis.py)  
Git blob: `86895e5ca70b5ee7dd4cc9dd474a004c427e750d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/SVC/Normal/logs)

SVC Normal EXP-002: inspect saved EXP-001 predictions, without retraining or Test access.



In [ ]:
%%archive_source 05_model_optimize/SVC/Normal/scripts/EXP_002_SVC_Normal_Error_Analysis.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""SVC Normal EXP-002: inspect saved EXP-001 predictions, without retraining or Test access."""
from __future__ import annotations
import csv
import json
from collections import Counter
from pathlib import Path

ROOT = Path(__file__).resolve().parents[4]
BASE = ROOT / '05_model_optimize' / 'SVC' / 'Normal' / 'results'
SRC = BASE / 'EXP_001_SVC_Normal_Baseline'
OUT = BASE / 'EXP_002_SVC_Normal_Error_Analysis'
LABELS = tuple('0123456789ABCDEF')


def write_csv(path, rows, cols):
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)


def main():
    path = SRC / 'handmade_predictions.csv'
    if not path.exists():
        raise FileNotFoundError(f'EXP-001 result missing: {path}')
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    if len(rows) != 2560:
        raise ValueError(f'Expected 2560 Hand-made predictions, found {len(rows)}')
    counter = Counter()
    writer_counter = Counter()
    label_total = Counter()
    label_errors = Counter()
    by_writer = {}
    for row in rows:
        w = int(row['writer'])
        t, p = row['true'], row['prediction']
        if w not in (41, 42, 43, 44) or t not in LABELS or p not in LABELS:
            raise ValueError(f'Invalid prediction row: {row}')
        label_total[t] += 1
        by_writer.setdefault(w, [0, 0])
        by_writer[w][0] += 1
        if t != p:
            counter[(t, p)] += 1
            writer_counter[w] += 1
            label_errors[t] += 1
        else:
            by_writer[w][1] += 1
    if sorted(by_writer) != [41, 42, 43, 44] or any(n != 640 for n, _ in by_writer.values()):
        raise ValueError('Writer counts are inconsistent')
    if any(label_total[c] != 160 for c in LABELS):
        raise ValueError('Each class must have exactly 160 Hand-made examples')
    pairs = [{'true': t, 'predicted': p, 'count': n}
             for (t, p), n in sorted(counter.items(), key=lambda x: (-x[1], x[0][0], x[0][1]))]
    classes = [{'class': c, 'total': label_total[c], 'errors': label_errors[c],
                'correct': label_total[c] - label_errors[c],
                'recall': (label_total[c] - label_errors[c]) / label_total[c]}
               for c in LABELS]
    writers = [{'writer': w, 'total': n, 'correct': ok,
                'errors': writer_counter[w], 'accuracy': ok / n}
               for w, (n, ok) in sorted(by_writer.items())]
    OUT.mkdir(parents=True, exist_ok=True)
    write_csv(OUT / 'confusion_pairs.csv', pairs, ['true', 'predicted', 'count'])
    write_csv(OUT / 'per_class_error.csv', classes, ['class', 'total', 'errors', 'correct', 'recall'])
    write_csv(OUT / 'writer_error.csv', writers, ['writer', 'total', 'correct', 'errors', 'accuracy'])
    summary = {'experiment': 'EXP-002 SVC Normal error analysis', 'source': str(path.relative_to(ROOT)),
               'handmade_n': len(rows), 'correct': sum(r['correct'] for r in writers),
               'errors': sum(r['errors'] for r in writers), 'top_confusions': pairs[:12],
               'weakest_classes': sorted(classes, key=lambda x: (x['recall'], x['class']))[:8],
               'writers': writers, 'model_retrained': False, 'official_test_used': False,
               'evaluation_role': 'Repeated-use Hand-made development; not independent test'}
    (OUT / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print('SVC Normal EXP-002 — Error analysis (no retraining)', flush=True)
    print(f"Hand-made={len(rows)}, correct={summary['correct']}, errors={summary['errors']}")
    print('TOP CONFUSIONS (true -> predicted):')
    for r in pairs[:10]:
        print(f"  {r['true']} -> {r['predicted']}: {r['count']}")
    print('WEAKEST CLASSES:')
    for r in summary['weakest_classes'][:8]:
        print(f"  {r['class']}: recall={r['recall']:.4f}, errors={r['errors']}/160")
    print('Saved:', OUT)


if __name__ == '__main__':
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")